# Обучение модели Diabetes (sklearn)

ДЗ по теме 24: Полноценный ML-сервис

Задачи:
1. Обучить RandomForestRegressor на датасете load_diabetes
2. Использовать StandardScaler в пайплайне
3. Зарегистрировать модель в MLflow под именем "diabets"
4. Сохранить модель для использования в ML-сервисе

In [ ]:
import os
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Подключение к MLflow (localhost при локальном запуске, mlflow при запуске в docker-compose)
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f"MLflow URI: {MLFLOW_TRACKING_URI}")

In [ ]:
# Загрузка датасета diabetes (10 признаков)
data = load_diabetes(return_X_y=False)
X, y = data.data, data.target
feature_names = data.feature_names
print(f"Признаки: {feature_names}")
print(f"Размер: X={X.shape}, y={y.shape}")

In [ ]:
# Разбиение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Создание пайплайна: StandardScaler + RandomForestRegressor
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)

In [ ]:
# Оценка модели
y_pred = pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MSE: {mse:.2f}")
print(f"R²: {r2:.4f}")

In [ ]:
# Регистрация в MLflow
EXPERIMENT_NAME = "diabetes-experiment"
REGISTERED_MODEL_NAME = "diabets"

try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
with mlflow.start_run(run_name="diabets-rf-v1") as run:
    run_id = run.info.run_id
    # Логирование параметров
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("n_features", X.shape[1])
    
    # Логирование метрик
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    
    # Логирование модели (без registered_model_name — совместимо с MLflow 2.11 сервером и 3.x клиентом)
    # Используем runs:/run_id/model вместо models:/diabets/1
    mlflow.sklearn.log_model(pipeline, "model")
    
    # Регистрация в Model Registry (только для MLflow 2.11 клиента)
    try:
        mlflow.register_model(f"runs:/{run_id}/model", REGISTERED_MODEL_NAME)
        print(f"Модель зарегистрирована в Registry: {REGISTERED_MODEL_NAME}")
    except Exception:
        print(f"Registry пропущен (клиент 3.x + сервер 2.11). Загрузка по runs:/{run_id}/model")
    
    print(f"Run ID: {run_id}")

In [ ]:
# Скачивание модели для использования в ML-сервисе
# Используем runs:/run_id/model (артефакт run), т.к. Model Registry недоступен при MLflow 3.x клиент + 2.11 сервер

model_uri = f"runs:/{run.info.run_id}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)
print(f"Модель загружена: {model_uri}")

# Сохранение модели в локальную папку для Docker (research/ -> hw_topic24/models/diabets)
PARENT_DIR = os.path.dirname(os.getcwd())  # hw_topic24
MODEL_SAVE_PATH = os.path.join(PARENT_DIR, "models", "diabets")
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
mlflow.sklearn.save_model(loaded_model, MODEL_SAVE_PATH)
print(f"Модель сохранена в: {MODEL_SAVE_PATH}")

In [ ]:
# Проверка предсказания
test_sample = X_test[:1]
pred = loaded_model.predict(test_sample)
print(f"Тестовое предсказание: {pred[0]:.2f}")
print(f"Истинное значение: {y_test[0]:.2f}")